# Lab · Cross-entropy và khởi tạo xấu · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 6, Bài 2). Theo hai lab của khóa ("Logistic Regression and Bad Initialization Value" và "Logistic Regression Cross Entropy"): huấn luyện logistic regression từ một khởi tạo xấu với **MSE** rồi với **cross-entropy**, và so sánh.

Lưu ý: trang lab Cross Entropy gốc của khóa không có đường link tải file, nên phần cross-entropy ở đây được viết lại dựa trên video và lab MSE.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. Dữ liệu: nhãn 1 khi x > 0.2

Viết lớp `Data(Dataset)`: `self.x = torch.arange(-1, 1, 0.1).view(-1, 1)`; `self.y = torch.zeros(self.x.shape[0], 1)`; rồi đặt `self.y[self.x[:, 0] > 0.2] = 1`; `self.len`; `__getitem__`, `__len__` như thường lệ.
Tạo `data_set = Data()`. Lưu `n_pos` = số mẫu có nhãn 1. Tính tay trước: những `x` nào lớn hơn 0.2?

In [ ]:
class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        self.y = torch.zeros(self.x.shape[0], 1)
        self.y[self.x[:, 0] > 0.2] = 1
        self.len = self.x.shape[0]

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len

data_set = Data()
n_pos = int(data_set.y.sum().item())
print(len(data_set), n_pos)

In [ ]:
assert len(data_set) == 20
assert n_pos == 7, "7 mẫu có x > 0.2"
print("✅ Bài 1 đúng (13 mẫu lớp 0, 7 mẫu lớp 1)")

## Bài 2. Mô hình và khởi tạo tham số

Viết lớp `logistic_regression(nn.Module)` (`self.linear = nn.Linear(n_inputs, 1)`; `forward` trả `torch.sigmoid(self.linear(x))`).
Viết hàm `make_model(w0, b0)` trả về mô hình `logistic_regression(1)` với `weight = w0`, `bias = b0` (đặt bằng `torch.no_grad()` và `copy_`).
Lưu `bad = make_model(-5.0, -10.0)` (khởi tạo "xấu") và `good = make_model(0.0, 0.0)`.

In [ ]:
class logistic_regression(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.linear = nn.Linear(n_inputs, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

def make_model(w0, b0):
    m = logistic_regression(1)
    with torch.no_grad():
        m.linear.weight.copy_(torch.tensor([[w0]]))
        m.linear.bias.copy_(torch.tensor([b0]))
    return m

bad = make_model(-5.0, -10.0)
good = make_model(0.0, 0.0)

In [ ]:
assert bad.linear.weight.item() == -5.0 and bad.linear.bias.item() == -10.0
assert good.linear.weight.item() == 0.0 and good.linear.bias.item() == 0.0
print("✅ Bài 2 đúng")

## Bài 3. Gradient lúc đầu: MSE so với cross-entropy

Với mô hình `bad` (chưa huấn luyện), tính loss trên **toàn bộ** `data_set.x`, `data_set.y` bằng hai cách rồi `backward()`:
- `nn.MSELoss()` → lưu `g_mse = abs(gradient của weight)`;
- `nn.BCELoss()` → lưu `g_bce = abs(gradient của weight)`.
Dùng một bản sao mô hình mới (gọi `make_model(-5., -10.)`) cho mỗi lần để gradient không dồn vào nhau.

**Đoán trước:** gradient nào lớn hơn, và lớn hơn bao nhiêu bậc?

In [ ]:
def grad_for(criterion):
    m = make_model(-5.0, -10.0)
    loss = criterion(m(data_set.x), data_set.y)
    loss.backward()
    return abs(m.linear.weight.grad.item())

g_mse = grad_for(nn.MSELoss())
g_bce = grad_for(nn.BCELoss())
print(g_mse, g_bce)

In [ ]:
assert g_mse < 1e-3, "MSE: gradient gần như bằng 0 (vùng phẳng)"
assert g_bce > 0.1, "cross-entropy: gradient vẫn đủ lớn để học"
assert g_bce / max(g_mse, 1e-12) > 1000
print("✅ Bài 3 đúng: chỗ này MSE không biết đi hướng nào, còn cross-entropy thì biết")

## Bài 4. Hàm huấn luyện chung

Viết `train(model, criterion, lr=2.0, epochs=100, batch_size=3)`:
- `DataLoader(data_set, batch_size=batch_size)` và `torch.optim.SGD(model.parameters(), lr=lr)`;
- vòng lặp huấn luyện chuẩn;
- trả về `(loss_first, loss_last, accuracy)`: loss trên **toàn bộ** dữ liệu **trước** khi huấn luyện, loss trên toàn bộ dữ liệu **sau** huấn luyện, và độ chính xác trên toàn bộ dữ liệu (nhãn dự đoán `p >= 0.5`, so với `data_set.y`).
Tính loss và accuracy dưới `torch.no_grad()`.

In [ ]:
def train(model, criterion, lr=2.0, epochs=100, batch_size=3):
    loader = DataLoader(data_set, batch_size=batch_size)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    with torch.no_grad():
        loss_first = criterion(model(data_set.x), data_set.y).item()
    for epoch in range(epochs):
        for x, y in loader:
            loss = criterion(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        yhat = model(data_set.x)
        loss_last = criterion(yhat, data_set.y).item()
        accuracy = ((yhat >= 0.5).float() == data_set.y).float().mean().item()
    return loss_first, loss_last, accuracy

In [ ]:
_m = make_model(0.0, 0.0)
_a, _b, _acc = train(_m, nn.BCELoss(), lr=1.0, epochs=5)
assert _b < _a, "sau huấn luyện loss phải giảm"
assert 0.0 <= _acc <= 1.0
print("✅ Bài 4 đúng")

## Bài 5. MSE từ khởi tạo xấu

Huấn luyện mô hình khởi tạo xấu bằng **MSE**: `m_mse = make_model(-5., -10.)`, `r_mse = train(m_mse, nn.MSELoss())`.
Lưu `acc_mse = r_mse[2]` và `loss_change = abs(r_mse[0] - r_mse[1])`.
**Đoán trước:** độ chính xác bao nhiêu? (Nếu mô hình đoán toàn lớp 0 thì đúng bao nhiêu phần trăm?)

In [ ]:
m_mse = make_model(-5.0, -10.0)
r_mse = train(m_mse, nn.MSELoss())
acc_mse = r_mse[2]
loss_change = abs(r_mse[0] - r_mse[1])
print(r_mse)

In [ ]:
assert acc_mse < 0.7, "MSE từ khởi tạo xấu chỉ đạt khoảng 65%, tức là đoán toàn lớp 0 (13/20)"
assert loss_change < 0.01, "loss gần như không đổi: mô hình kẹt"
print("✅ Bài 5 đúng: mô hình kẹt")

## Bài 6. Cross-entropy từ cùng khởi tạo xấu

Lặp lại bài 5 nhưng với **`nn.BCELoss()`** (mô hình mới khởi tạo `-5, -10`). Lưu `r_bce`, `acc_bce` và `loss_drop = r_bce[0] - r_bce[1]`.
**Đoán trước** rồi so với bài 5: khác nhau ở đâu?

In [ ]:
m_bce = make_model(-5.0, -10.0)
r_bce = train(m_bce, nn.BCELoss())
acc_bce = r_bce[2]
loss_drop = r_bce[0] - r_bce[1]
print(r_bce)

In [ ]:
assert acc_bce == 1.0, "cross-entropy thoát khỏi khởi tạo xấu và phân loại đúng cả 20 mẫu"
assert loss_drop > 2.0
print("✅ Bài 6 đúng: cùng dữ liệu, cùng khởi tạo, cùng optimizer; chỉ khác loss")

## Bài 7. MSE có luôn thất bại không?

Huấn luyện bằng **MSE** nhưng từ khởi tạo **tốt**: `make_model(0., 0.)`. Lưu `acc_mse_good`.
Câu hỏi: kết quả nói gì về nguyên nhân thất bại ở bài 5: do MSE nói chung hay do khởi tạo kết hợp với MSE?

In [ ]:
r_good = train(make_model(0.0, 0.0), nn.MSELoss())
acc_mse_good = r_good[2]
print(r_good)

In [ ]:
assert acc_mse_good == 1.0
print("✅ Bài 7 đúng: MSE học được khi khởi tạo tốt; nó rất NHẠY với khởi tạo vì có vùng gradient biến mất")

## Bài 8. Cross-entropy bằng tay

`p = [0.9, 0.1, 0.8]` (xác suất dự đoán), `y = [1., 1., 0.]`.
- `bce_manual = -mean( y·log p + (1−y)·log(1−p) )` bằng tensor;
- `bce_torch = nn.BCELoss()(p, y)`;
- `small` = loss một mẫu khi `y = 1, p = 0.9`; `big` = loss một mẫu khi `y = 1, p = 0.1`; `ratio = big / small`.

In [ ]:
p = torch.tensor([0.9, 0.1, 0.8])
y = torch.tensor([1.0, 1.0, 0.0])
bce_manual = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()
bce_torch = nn.BCELoss()(p, y)
small = -torch.log(torch.tensor(0.9)).item()
big = -torch.log(torch.tensor(0.1)).item()
ratio = big / small
print(bce_manual, bce_torch, ratio)

In [ ]:
assert abs(bce_manual.item() - 1.3391) < 1e-3
assert torch.isclose(bce_manual, bce_torch)
assert abs(small - 0.1054) < 1e-3 and abs(big - 2.3026) < 1e-3
assert ratio > 20, "tự tin mà sai bị phạt nặng hơn hơn 20 lần"
print("✅ Bài 8 đúng")

## Hết